# Step 3: Dynamic Few-Shot Information Extraction

This notebook demonstrates the **dynamic few-shot prompting feature** for Pattern 2. It shows how to:

- Configure dynamic few-shot Lambda functions extraction
- Compare default vs examples-enhanced extraction results
- Inspect Lambda payloads and responses
- Handle errors and monitor performance

**Prerequisites:**
- Completed Step 2 (Classification)
- AWS Lambda permissions to create/invoke functions
- Dynamic few-shot Lambda function deployed
- S3 Vectors index populated with examples (`notebooks/misc/fewshot_dataset_import.ipynb`)

**Key Feature:**
The `dynamic_few_shot_lambda_arn` configuration field allows you to dynamically retrieve similar examples using S3 Vectors similarity search to improve extraction accuracy through few-shot prompting.

## 1. Setup and Import Libraries

In [ ]:
import os
import json
import time
import logging
import boto3
from pathlib import Path
import yaml

# Import IDP libraries
from idp_common.models import Document, Status
from idp_common.s3 import get_json_content
from idp_common import extraction

# Configure logging to see Lambda invocation details
logging.basicConfig(level=logging.INFO)
logging.getLogger('idp_common.extraction').setLevel(logging.INFO)
logging.getLogger('idp_common.bedrock.client').setLevel(logging.INFO)

print("Libraries imported successfully")

## 2. Load Previous Step Data

In [ ]:
# Load document from previous step
classification_data_dir = Path(".data/step2_classification")

# Load document object from JSON
document_path = classification_data_dir / "document.json"
with open(document_path, 'r') as f:
    document = Document.from_json(f.read())

# Load configuration directly from config files
config_dir = Path("config")
CONFIG = {}

# Load each configuration file
config_files = [
    "extraction_with_few_shot.yaml",
    "classes.yaml"
]

for config_file in config_files:
    config_path = config_dir / config_file
    if config_path.exists():
        with open(config_path, 'r') as f:
            file_config = yaml.safe_load(f)
            CONFIG.update(file_config)
        print(f"Loaded {config_file}")
    else:
        print(f"Warning: {config_file} not found")

# Load environment info
env_path = classification_data_dir / "environment.json"
with open(env_path, 'r') as f:
    env_info = json.load(f)

# Set environment variables
os.environ['AWS_REGION'] = env_info['region']
os.environ['METRIC_NAMESPACE'] = 'IDP-Dynamic-Few-Shot'

print(f"Loaded document: {document.id}")
print(f"Document status: {document.status.value}")
print(f"Number of sections: {len(document.sections) if document.sections else 0}")
print(f"Loaded configuration sections: {list(CONFIG.keys())}")

## 3. Configure Dynamic Few-Shot Lambda ARN

In [ ]:
# 🔧 CONFIGURATION: Set your dynamic few-shot Lambda ARN here
# Replace with your actual Lambda function ARN for live testing

# Check if dynamic few-shot Lambda function exists
lambda_client = boto3.client('lambda')
DYNAMIC_FEW_SHOT_LAMBDA_ARN = None

try:
    response = lambda_client.get_function(FunctionName='GENAIIDP-dynamic-few-shot')
    DYNAMIC_FEW_SHOT_LAMBDA_ARN = response['Configuration']['FunctionArn']
    print(f"✅ Found dynamic few-shot Lambda function: {DYNAMIC_FEW_SHOT_LAMBDA_ARN}")
except lambda_client.exceptions.ResourceNotFoundException:
    print("⚠️  Dynamic Few-Shot Lambda function not found: GENAIIDP-dynamic-few-shot")
    print("💡 Deploy using: cd notebooks/examples/dynamic-few-shot-lambda && sam deploy --guided")
except Exception as e:
    print(f"Error checking Lambda function: {e}")

if not DYNAMIC_FEW_SHOT_LAMBDA_ARN:
    print("⚠️  No dynamic few-shot Lambda ARN configured")
    print("💡 This demo will show standard extraction without few-shot examples")
    print("🔧 To test with examples, deploy the dynamic few-shot Lambda first")
else:
    print(f"✅ Dynamic few-shot Lambda ARN configured: {DYNAMIC_FEW_SHOT_LAMBDA_ARN}")
    print("🚀 This demo will use few-shot examples from S3 Vectors")

## 4. Extraction Comparison: Default vs Dynamic Few-Shot

### 4.1 Default Extraction (Without Dynamic Few-Shot)

In [ ]:
# Create configuration WITHOUT dynamic few-shot Lambda
config_default = CONFIG.copy()
if 'dynamic_few_shot_lambda_arn' in config_default.get('extraction', {}):
    del config_default['extraction']['dynamic_few_shot_lambda_arn']

print("=== DEFAULT EXTRACTION CONFIGURATION ===")
print(f"Model: {config_default.get('extraction', {}).get('model')}")
print(f"Dynamic Few-Shot Lambda: {config_default.get('extraction', {}).get('dynamic_few_shot_lambda_arn', 'None')}")

# Create extraction service with default config
extraction_service_default = extraction.ExtractionService(config=config_default)
print("\n✅ Default extraction service initialized")

In [ ]:
# Run default extraction on first section
if document.sections:
    first_section = document.sections[0]
    print(f"🔄 Processing section {first_section.section_id} with DEFAULT prompts")
    print(f"Classification: {first_section.classification}")
    print(f"Pages: {first_section.page_ids}")
    
    # Save original document state
    document_default = Document.from_json(document.to_json())
    
    # Process with default extraction
    start_time = time.time()
    document_default = extraction_service_default.process_document_section(
        document=document_default,
        section_id=first_section.section_id
    )
    default_extraction_time = time.time() - start_time
    
    print(f"✅ Default extraction completed in {default_extraction_time:.2f} seconds")

    # Store results for comparison
    default_section_result = None
    for section in document_default.sections:
        if section.section_id == first_section.section_id:
            default_section_result = section
            break
            
else:
    print("⚠️ No sections found in document")

In [ ]:
# Show section extraction result
if default_section_result:
    print(f"\nSection {default_section_result.section_id} extraction result:")
    extraction_result_uri = default_section_result.extraction_result_uri

    if extraction_result_uri:
        result = get_json_content(extraction_result_uri)
        result_json = json.dumps(result["inference_result"], indent=2)
        print(result_json)

else:
    print("⚠️ No sections found in document")

### 4.2 Dynamic Few-Shot Extraction using Lambda

In [ ]:
if DYNAMIC_FEW_SHOT_LAMBDA_ARN:
    # Create configuration WITH dynamic few-shot Lambda
    config_few_shot = CONFIG.copy()
    config_few_shot['extraction']['dynamic_few_shot_lambda_arn'] = DYNAMIC_FEW_SHOT_LAMBDA_ARN
    
    print("=== DYNAMIC FEW-SHOT EXTRACTION CONFIGURATION ===")
    print(f"Model: {config_few_shot.get('extraction', {}).get('model')}")
    print(f"Dynamic Few-Shot Lambda: {DYNAMIC_FEW_SHOT_LAMBDA_ARN}")
    print(f"Lambda Function Name: {DYNAMIC_FEW_SHOT_LAMBDA_ARN.split(':')[-1]}")
    
    # Create extraction service with dynamic few-shot config
    extraction_service_few_shot = extraction.ExtractionService(config=config_few_shot)
    
    print("\n✅ Dynamic few-shot extraction service initialized")
    
else:
    print("⚠️ No dynamic few-shot Lambda ARN configured - skipping demonstration")
    config_few_shot = None
    extraction_service_few_shot = None

In [ ]:
# Run dynamic few-shot extraction on first section
if DYNAMIC_FEW_SHOT_LAMBDA_ARN and document.sections:
    first_section = document.sections[0]
    print(f"🔄 Processing section {first_section.section_id} with DYNAMIC FEW-SHOT")
    print(f"Classification: {first_section.classification}")
    print(f"Pages: {first_section.page_ids}")
    
    # Create fresh document copy for examples processing
    document_few_shot = Document.from_json(document.to_json())
    
    # Process with dynamic few-shot extraction
    start_time = time.time()
    
    try:
        document_few_shot = extraction_service_few_shot.process_document_section(
            document=document_few_shot,
            section_id=first_section.section_id
        )
        few_shot_extraction_time = time.time() - start_time
        
        print(f"✅ Dynamic few-shot extraction completed in {few_shot_extraction_time:.2f} seconds")
        
        # Store results for comparison
        few_shot_section_result = None
        for section in document_few_shot.sections:
            if section.section_id == first_section.section_id:
                few_shot_section_result = section
                break
                
        # Performance comparison
        overhead = few_shot_extraction_time - default_extraction_time
        print(f"\n📊 Performance Comparison:")
        print(f"   Default: {default_extraction_time:.2f}s")
        print(f"   Dynamic Few-Shot: {few_shot_extraction_time:.2f}s")
        print(f"   Dynamic Few-Shot Overhead: {overhead:.2f}s ({overhead/default_extraction_time*100:.1f}% increase)")
        
    except Exception as e:
        print(f"❌ Dynamic few-shot extraction failed: {e}")
        print("\n🔍 This demonstrates the fail-fast error handling behavior")
        few_shot_section_result = None
        few_shot_extraction_time = None
        
else:
    print("⚠️ Skipping dynamic few-shot extraction (no Lambda configured or no sections)")
    document_few_shot = None
    few_shot_section_result = None
    few_shot_extraction_time = None

In [ ]:
# Show section extraction result
if few_shot_section_result:
    print(f"\nSection {few_shot_section_result.section_id} extraction result:")
    extraction_result_uri = few_shot_section_result.extraction_result_uri

    if extraction_result_uri:
        result = get_json_content(extraction_result_uri)
        result_json = json.dumps(result["inference_result"], indent=2)
        print(result_json)

else:
    print("⚠️ No sections found in document")

## 5. Results and Summary

In [ ]:
print("=== DEMO COMPLETE: SUMMARY ===")

sections_processed = 1 if document.sections else 0
dynamic_few_shot_used = DYNAMIC_FEW_SHOT_LAMBDA_ARN is not None

print(f"\n✅ DEMO RESULTS:")
print(f"   📄 Document processed: {document.id}")
print(f"   📊 Sections processed: {sections_processed}")
print(f"   🔧 Dynamic Few-Shot used: {'Yes' if dynamic_few_shot_used else 'No'}")

if dynamic_few_shot_used and 'few_shot_extraction_time' in locals() and examples_extraction_time:
    print(f"   ⏱️  Performance overhead: {few_shot_extraction_time - default_extraction_time:.2f}s")
    print(f"   📈 Accuracy improvement: Enhanced with few-shot examples")

print(f"\n🚀 TO IMPLEMENT DYNAMIC FEW-SHOT IN PRODUCTION:")
print(f"   1. 📝 Deploy dynamic few-shot Lambda stack")
print(f"   2. 📊 Populate S3 Vectors index with example documents")
print(f"   3. ⚙️  Add 'dynamic_few_shot_lambda_arn' to extraction config")
print(f"   4. 🧪 Test with your actual documents and use cases")
print(f"   5. 📊 Monitor CloudWatch logs for performance and accuracy")

print(f"\n📚 RESOURCES:")
print(f"   📖 Documentation: notebooks/examples/dynamic-few-shot-lambda/README.md")
print(f"   🔧 Lambda Function: notebooks/examples/dynamic-few-shot-lambda/GENAIIDP-dynamic-few-shot.py")
print(f"   ☁️  Deploy: cd notebooks/examples/dynamic-few-shot-lambda && sam deploy --guided")
print(f"   📊 Import Dataset: notebooks/misc/fewshot_dataset_import.ipynb")

print(f"\n📌 CONTINUE TO: step4_assessment.ipynb")